# Closure Weather Label Construction

This notebook builds the supervised learning dataset for forecasting I-80 Donner Summit road closures.

The raw inputs are:
- manually labeled closure and reopening timestamps from historical road closure posts
- hourly weather observations for the Donner Summit / I-80 corridor

Each row in the final dataset represents **one hour** of weather conditions, along with labels for whether a **new closure event** will begin within the next 6, 12, or 24 hours.

This notebook:
- converts raw closure posts into closure intervals
- resolves missing reopening times conservatively
- merges closure intervals onto hourly weather observations
- defines event-level closure starts to avoid overcounting long closures
- creates forecasting targets for downstream modeling

The result is a clean, time-aligned hourly dataset designed for road-closure prediction.

## Data Sources

This notebook combines two data sources:

1. **Manual closure timeline**
   A manually curated dataset of I-80 Donner Summit closure and reopening timestamps extracted from historical road closure posts. These records define when closures began and, when available, when they ended.

2. **Hourly weather observations**
   Historical hourly weather measurements for the Donner Summit / I-80 corridor, including snowfall, precipitation, wind, temperature, pressure, and related conditions.

The weather table serves as the base hourly timeline. Closure records are first converted into time intervals, then mapped onto hourly weather rows to create supervised learning labels.

The final merged dataset is an hourly panel of weather conditions with road-closure outcome variables for downstream forecasting.

## Load Packages and Raw Data

This section imports the packages and helper functions used in the labeling pipeline, then loads the two raw inputs:

- manually annotated closure / reopening records
- hourly historical weather observations for Donner Summit

For reproducibility, these files should be loaded from repo-relative paths so the notebook can run on another machine without editing file locations.

In [26]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]   # road_project
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from main import utils
import pandas as pd
import numpy as np
from pathlib import Path

from main import utils

CLOSURE_PATH = PROJECT_ROOT / "facebook" / "data" / "manual_closure_or_reopening_times_finished.xlsx"
WEATHER_PATH = PROJECT_ROOT / "weather" / "historical_weather_date.csv"

closure_df = pd.read_excel(CLOSURE_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

weather_df = weather_df.drop(columns=["Unnamed: 0"], errors="ignore")

# Weather timestamps: parse and keep timezone awareness
weather_df["date"] = pd.to_datetime(weather_df["date"], utc=True, errors="coerce")

# Closure timestamps: parse first
closure_df["timestamp"] = pd.to_datetime(closure_df["timestamp"], errors="coerce")
closure_df["time_reopened"] = pd.to_datetime(closure_df["time_reopened"], errors="coerce")

# If closure times were recorded in local California time, localize them first
closure_df["timestamp"] = closure_df["timestamp"].dt.tz_localize("America/Los_Angeles")
closure_df["time_reopened"] = closure_df["time_reopened"].dt.tz_localize("America/Los_Angeles")

# Convert closure times to the same timezone as weather
closure_df["timestamp"] = closure_df["timestamp"].dt.tz_convert("UTC")
closure_df["time_reopened"] = closure_df["time_reopened"].dt.tz_convert("UTC")

In [6]:
print(weather_df["date"].dtype)
print(closure_df["timestamp"].dtype)
print(closure_df["time_reopened"].dtype)

datetime64[us, UTC]
datetime64[us, UTC]
datetime64[us, UTC]


## Standardize Timestamps

The closure and weather data come from different sources and must be standardized before any interval construction or labeling.

In particular, both timestamp columns are parsed as timezone-aware datetimes and converted to the same timezone so that closure events line up correctly with hourly weather observations.

Because the weather table is hourly, all downstream labels are defined relative to hourly timestamps.

In [7]:
print("closure_df shape:", closure_df.shape)
print("weather_df shape:", weather_df.shape)

print("\nclosure columns:")
print(closure_df.columns.tolist())

print("\nweather columns:")
print(weather_df.columns.tolist())

# Light validation checks
assert "timestamp" in closure_df.columns
assert "time_reopened" in closure_df.columns
assert "date" in weather_df.columns

assert "UTC" in str(closure_df["timestamp"].dtype)
assert "UTC" in str(weather_df["date"].dtype)

closure_df shape: (617, 22)
weather_df shape: (79176, 23)

closure columns:
['Unnamed: 0.1', 'Unnamed: 0', 'facebookId', 'facebookUrl', 'feedbackId', 'likes', 'postId', 'timestamp', 'user/name', 'user/profilePic', 'user/profileUrl', 'viewsCount', 'time', 'text', 'is_wb', 'is_eb', 'is_closure', 'is_reopening', 'is_placing', 'is_lifting', 'URL', 'time_reopened']

weather columns:
['date', 'temperature_2m', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_high', 'cloud_cover_mid', 'wind_direction_100m', 'wind_direction_10m', 'wind_speed_100m', 'wind_speed_10m', 'rain', 'snowfall', 'snow_depth', 'weather_code', 'pressure_msl', 'surface_pressure', 'precipitation', 'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m', 'is_day', 'snow_depth_water_equivalent', 'sunshine_duration']


In [8]:
print("weather dtype:", weather_df["date"].dtype)
print("closure timestamp dtype:", closure_df["timestamp"].dtype)
print("closure reopening dtype:", closure_df["time_reopened"].dtype)

print("\nweather sample:")
display(weather_df[["date"]].head())

print("\nclosure sample:")
display(closure_df[["timestamp", "time_reopened"]].head())

weather dtype: datetime64[us, UTC]
closure timestamp dtype: datetime64[us, UTC]
closure reopening dtype: datetime64[us, UTC]

weather sample:


,date
0,2017-02-21 00:00:00+00:00
1,2017-02-21 01:00:00+00:00
2,2017-02-21 02:00:00+00:00
3,2017-02-21 03:00:00+00:00
4,2017-02-21 04:00:00+00:00



closure sample:


,timestamp,time_reopened
0,2026-02-20 15:32:55+00:00,NaT
1,2026-02-20 08:13:08+00:00,NaT
2,2026-02-19 18:27:26+00:00,2026-02-20 08:10:00+00:00
3,2026-02-19 15:04:09+00:00,2026-02-20 08:10:00+00:00
4,2026-02-19 00:27:05+00:00,2026-02-20 08:10:00+00:00


In [9]:
print("weather NaT count:", weather_df["date"].isna().sum())
print("closure timestamp NaT count:", closure_df["timestamp"].isna().sum())
print("reopening NaT count:", closure_df["time_reopened"].isna().sum())

weather NaT count: 0
closure timestamp NaT count: 0
reopening NaT count: 446


## Build Raw Closure Intervals

Each closure record is first converted into a start/end interval.

- If a reopening time is available, the interval runs from the recorded closure time to the recorded reopening time.
- If a reopening time is missing, only the closure hour is treated as confirmed closed. The next 23 hours are marked as ambiguous rather than assumed open or closed. This heuristic was chosen because it will allow unclear reopenings to not negatively affect model training. Some closures last long, but are not accounted for, this may mislead a model in training and any negatives are heavily outshined by the gains here.

This conservative rule avoids inventing closure duration when the reopening time is unknown. At this stage, the intervals remain close to the raw post-level records and have not yet been merged into larger closure events.

In [10]:
raw_intervals = utils.build_closure_intervals(
    closure_df,
    closure_col="timestamp",
    reopen_col="time_reopened",
    missing_reopen_hours=24
)

print("raw_intervals shape:", raw_intervals.shape)
display(raw_intervals.head())

# Validation checks
assert {"closure_start", "closure_end"}.issubset(raw_intervals.columns)
assert (raw_intervals["closure_end"] >= raw_intervals["closure_start"]).all()
assert raw_intervals["closure_start"].notna().all()
assert raw_intervals["closure_end"].notna().all()

if "has_reopening_time" in raw_intervals.columns:
    print("\nIntervals with reopening time:", raw_intervals["has_reopening_time"].sum())
    print("Intervals without reopening time:", (~raw_intervals["has_reopening_time"]).sum())
else:
    print("\nSource rows with reopening time:", closure_df["time_reopened"].notna().sum())
    print("Source rows without reopening time:", closure_df["time_reopened"].isna().sum())

raw_intervals shape: (617, 3)


,closure_start,closure_end,has_reopening_time
0,2017-02-22 02:56:34+00:00,2017-02-22 05:25:00+00:00,True
1,2017-02-22 05:25:26+00:00,2017-02-23 05:25:26+00:00,False
2,2017-02-22 06:12:23+00:00,2017-02-23 06:12:23+00:00,False
3,2017-02-22 07:17:22+00:00,2017-02-23 07:17:22+00:00,False
4,2017-02-22 15:51:27+00:00,2017-02-23 15:51:27+00:00,False



Intervals with reopening time: 171
Intervals without reopening time: 446


## Collapse Raw Records into Event-Level Intervals

Raw closure records may include multiple posts or updates about the same real-world closure event.

To create event-level intervals for forecasting, nearby or overlapping raw intervals are consolidated into a single closure event when they appear to describe the same episode.

This prevents one prolonged closure from being counted multiple times simply because it generated several posts or status updates.

The result is a cleaner event-level view of closures, which is more appropriate for defining forecast targets than treating every raw record as an independent event.

In [11]:
gap_results = []

for gap in [3, 6, 12, 24]:
    event_intervals_test = utils.build_event_intervals(
        raw_intervals,
        max_gap_hours=gap
    )
    gap_results.append({
        "gap_hours": gap,
        "merged_event_count": len(event_intervals_test)
    })

gap_results_df = pd.DataFrame(gap_results)
display(gap_results_df)

,gap_hours,merged_event_count
0,3,212
1,6,210
2,12,202
3,24,193


In [12]:
event_intervals = utils.build_event_intervals(raw_intervals, max_gap_hours=12)

In [13]:
print("Chosen merge gap: 12 hours")
print("Number of event-level intervals:", len(event_intervals))
display(event_intervals.head())

Chosen merge gap: 12 hours
Number of event-level intervals: 202


,closure_start,closure_end
0,2017-02-22 02:56:34+00:00,2017-02-23 23:58:12+00:00
1,2017-03-04 00:57:16+00:00,2017-03-07 23:30:43+00:00
2,2017-03-10 21:28:11+00:00,2017-03-11 21:28:11+00:00
3,2017-03-24 21:34:50+00:00,2017-03-25 22:51:29+00:00
4,2017-04-13 21:28:55+00:00,2017-04-15 13:36:07+00:00


## Label Hourly Weather Rows with Closure Status

Closure intervals are mapped onto the hourly weather table to create an hourly road-status label.

The resulting `closure` variable has three possible values:

- `1` = confirmed closed
- `0` = confirmed open
- `NA` = ambiguous period following a closure with no recorded reopening time

Ambiguous rows are kept during label construction for transparency, then removed before supervised modeling so the training labels reflect only confirmed open or closed periods.

In [14]:
# Use raw_intervals here so ambiguous periods from missing reopen times are preserved
weather_closure = utils.apply_closure_to_weather(
    weather_df,
    raw_intervals,
    weather_time_col="date",
    closure_label_col="closure"
)

In [15]:
print("weather_closure shape:", weather_closure.shape)
print(weather_closure["closure"].value_counts(dropna=False))
display(weather_closure[["date", "closure"]].head())

weather_closure shape: (79176, 24)
closure
0.0    71747
NaN     6144
1.0     1285
Name: count, dtype: int64


,date,closure
0,2017-02-21 00:00:00+00:00,0.0
1,2017-02-21 01:00:00+00:00,0.0
2,2017-02-21 02:00:00+00:00,0.0
3,2017-02-21 03:00:00+00:00,0.0
4,2017-02-21 04:00:00+00:00,0.0


## Remove Ambiguous Rows for Modeling

Rows with ambiguous closure status are excluded from the supervised modeling dataset.

This conservative choice reduces label noise by avoiding unsupported assumptions about whether post-closure periods with missing reopening times should be treated as open or closed.

In [16]:
clean_df = weather_closure.dropna(subset=["closure"]).copy()
clean_df = clean_df.sort_values("date").reset_index(drop=True)

clean_df["year"] = clean_df["date"].dt.year
clean_df["month"] = clean_df["date"].dt.month

clean_df["winter"] = np.where(
    clean_df["month"] >= 10,
    clean_df["year"].astype(str) + "-" + (clean_df["year"] + 1).astype(str),
    (clean_df["year"] - 1).astype(str) + "-" + clean_df["year"].astype(str)
)

## Mark Event-Level Closure Starts

Forecasting targets should be based on the start of a closure event, not every hour within an ongoing closure.

Using the event-level interval table, each hourly row is marked with `closure_start = 1` if it corresponds to the beginning of a merged closure event.

In [17]:
start_hours = (
    pd.Series(event_intervals["closure_start"].dt.floor("h"))
    .drop_duplicates()
)

clean_df = clean_df.copy()
clean_df["closure_start"] = clean_df["date"].isin(start_hours).astype(int)

print("Rows:", len(clean_df))
print("Open rows:", (clean_df["closure"] == 0).sum())
print("Closed rows:", (clean_df["closure"] == 1).sum())
print("Closure starts:", clean_df["closure_start"].sum())
print("Closure-start rate:", clean_df["closure_start"].mean())

display(clean_df.groupby("winter")["closure_start"].sum().sort_index())

# Validation
assert clean_df["closure_start"].sum() == len(start_hours)
assert (clean_df.loc[clean_df["closure_start"] == 1, "closure"] == 1).all()

Rows: 73032
Open rows: 71747
Closed rows: 1285
Closure starts: 202
Closure-start rate: 0.0027659108336071858


winter
2016-2017    22
2017-2018    25
2018-2019    30
2019-2020    18
2020-2021    17
2021-2022    28
2022-2023    27
2023-2024    13
2024-2025    16
2025-2026     6
Name: closure_start, dtype: int64

In [18]:
event_intervals_check = event_intervals.copy()
event_intervals_check["start_hour"] = event_intervals_check["closure_start"].dt.floor("h")
event_intervals_check["year"] = event_intervals_check["start_hour"].dt.year
event_intervals_check["month"] = event_intervals_check["start_hour"].dt.month
event_intervals_check["winter"] = np.where(
    event_intervals_check["month"] >= 10,
    event_intervals_check["year"].astype(str) + "-" + (event_intervals_check["year"] + 1).astype(str),
    (event_intervals_check["year"] - 1).astype(str) + "-" + event_intervals_check["year"].astype(str)
)

print("Event-level starts by winter:")
display(event_intervals_check.groupby("winter").size().sort_index())

print("Hourly dataset closure_start by winter:")
display(clean_df.groupby("winter")["closure_start"].sum().sort_index())

Event-level starts by winter:


winter
2016-2017    22
2017-2018    25
2018-2019    30
2019-2020    18
2020-2021    17
2021-2022    28
2022-2023    27
2023-2024    13
2024-2025    16
2025-2026     6
dtype: int64

Hourly dataset closure_start by winter:


winter
2016-2017    22
2017-2018    25
2018-2019    30
2019-2020    18
2020-2021    17
2021-2022    28
2022-2023    27
2023-2024    13
2024-2025    16
2025-2026     6
Name: closure_start, dtype: int64

## Create Forecasting Targets

To support practical trip-planning decisions, forecasting targets are constructed for multiple horizons:

- `will_close_in_6h`
- `will_close_in_12h`
- `will_close_in_24h`

For a given horizon `k`, a row is labeled `1` if a **new closure event** starts within the next `k` hours.

These targets are defined relative to `closure_start`, so the model learns to predict the onset of a closure event rather than continued hours within an ongoing closure. Only rows that are currently open are treated as candidate warning examples.

In [19]:
forecast_df = clean_df.copy()

for h in [6, 12, 24]:
    forecast_df = utils.make_future_closure_target(
        forecast_df,
        time_col="date",
        start_col="closure_start",
        closure_col="closure",
        horizon_hours=h,
        open_only=True
    )

## Inspect Forecasting Targets

The 6-hour, 12-hour, and 24-hour targets represent different planning windows for Bay Area to Tahoe travel.

- `will_close_in_6h` captures short-notice risk
- `will_close_in_12h` captures same-day planning risk
- `will_close_in_24h` captures next-day trip planning risk

The next step checks how common each target is and verifies that longer-horizon targets behave consistently with shorter-horizon ones.

In [20]:
target_cols = ["will_close_in_6h", "will_close_in_12h", "will_close_in_24h"]

for col in target_cols:
    print(f"\n{col}")
    print(forecast_df[col].value_counts(dropna=False).sort_index())
    print("positive rate (non-null rows):", forecast_df[col].mean())

# Monotonicity check on rows where all targets are defined
mask = forecast_df[target_cols].notna().all(axis=1)

assert (forecast_df.loc[mask, "will_close_in_6h"] <= forecast_df.loc[mask, "will_close_in_12h"]).all()
assert (forecast_df.loc[mask, "will_close_in_12h"] <= forecast_df.loc[mask, "will_close_in_24h"]).all()

print("\nMonotonicity checks passed.")


will_close_in_6h
will_close_in_6h
0    71820
1     1212
Name: count, dtype: int64
positive rate (non-null rows): 0.016595465001643116

will_close_in_12h
will_close_in_12h
0    70608
1     2424
Name: count, dtype: int64
positive rate (non-null rows): 0.03319093000328623

will_close_in_24h
will_close_in_24h
0    68214
1     4818
Name: count, dtype: int64
positive rate (non-null rows): 0.06597108116989812

Monotonicity checks passed.


## Final Dataset Summary

This section checks the final class balance and event counts after label construction.

These summaries help verify that:
- the closure labels are reasonable
- closure starts are distributed across winters
- the forecast targets have plausible positive rates

In [21]:
summary = pd.DataFrame({
    "metric": [
        "rows",
        "open_rows",
        "closed_rows",
        "closure_hours_rate",
        "closure_starts",
        "closure_start_rate",
        "will_close_in_6h_positives",
        "will_close_in_6h_rate",
        "will_close_in_12h_positives",
        "will_close_in_12h_rate",
        "will_close_in_24h_positives",
        "will_close_in_24h_rate",
    ],
    "value": [
        len(forecast_df),
        (forecast_df["closure"] == 0).sum(),
        (forecast_df["closure"] == 1).sum(),
        (forecast_df["closure"] == 1).mean(),
        forecast_df["closure_start"].sum(),
        forecast_df["closure_start"].mean(),
        forecast_df["will_close_in_6h"].sum(),
        forecast_df["will_close_in_6h"].mean(),
        forecast_df["will_close_in_12h"].sum(),
        forecast_df["will_close_in_12h"].mean(),
        forecast_df["will_close_in_24h"].sum(),
        forecast_df["will_close_in_24h"].mean(),
    ]
})

display(summary)

,metric,value
0,rows,73032.000000
1,open_rows,71747.000000
2,closed_rows,1285.000000
3,closure_hours_rate,0.017595
4,closure_starts,202.000000
5,closure_start_rate,0.002766
6,will_close_in_6h_positives,1212.000000
7,will_close_in_6h_rate,0.016595
8,will_close_in_12h_positives,2424.000000
9,will_close_in_12h_rate,0.033191


## Save Cleaned Outputs

Save the cleaned hourly dataset so that later notebooks can focus on feature engineering, model selection, and evaluation without repeating the label-construction pipeline.

In [22]:
# Save cleaned outputs for later notebooks
output_dir = Path.cwd().resolve().parents[1] / "main" / "data"
output_dir.mkdir(parents=True, exist_ok=True)

# Main cleaned datasets
weather_closure.to_csv(output_dir / "weather_closure_labeled.csv", index=False)
clean_df.to_csv(output_dir / "weather_closure_clean.csv", index=False)
forecast_df.to_csv(output_dir / "weather_closure_forecast_targets.csv", index=False)

# Interval tables
raw_intervals.to_csv(output_dir / "raw_closure_intervals.csv", index=False)
event_intervals.to_csv(output_dir / "event_closure_intervals.csv", index=False)

print("Saved files to:", output_dir)
print(sorted([p.name for p in output_dir.glob("*.csv")]))

Saved files to: /Users/hudson/Desktop/road_project/main/data
['event_closure_intervals.csv', 'raw_closure_intervals.csv', 'weather_closure_clean.csv', 'weather_closure_forecast_targets.csv', 'weather_closure_labeled.csv']


## Key Outputs

This notebook produces five reusable artifacts for downstream modeling:

- `raw_closure_intervals.csv`: post-level closure intervals built from manually labeled closure and reopening records
- `event_closure_intervals.csv`: merged event-level closure intervals used to define closure starts
- `weather_closure_labeled.csv`: hourly weather rows labeled with road status (`closure`)
- `weather_closure_clean.csv`: confirmed-status hourly dataset with ambiguous rows removed
- `weather_closure_forecast_targets.csv`: hourly modeling dataset with `closure_start` and 6h / 12h / 24h forecasting targets

These saved outputs allow later notebooks to focus on feature engineering, baseline modeling, and evaluation without repeating the label-construction pipeline.